# Teil 1: Datenbereinigung und Aufbereitung von Game-Datensätzen

Data Analytics Project 2026 
Yasemin Senel 

Ziel: Zwei Datensätze sauber laden, bereinigen, dokumentieren und als neue CSV-Dateien exportieren.

Datensätze: 17k_strategy_games.csv und Highest_Grossing_Mobile_Game.csv

## Inhalt

1. Setup und Daten laden  
2. Erster Datencheck  
3. Hilfsfunktionen  
4. Bereinigung des Strategy-Games-Datensatzes  
5. Bereinigung des Highest-Grossing-Datensatzes  
6. Qualitätschecks  
7. Export der bereinigten Dateien  


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from IPython.display import display, Markdown

# Rohdaten laden
strategy_path = "Datensatz/17k_strategy_games.csv"
grossing_path = "Datensatz/Highest_Grossing_Mobile_Game.csv"

strategy_raw = pd.read_csv(strategy_path)
grossing_raw = pd.read_csv(grossing_path)

print("Strategy Games (roh):", strategy_raw.shape)
print("Highest Grossing (roh):", grossing_raw.shape)


Strategy Games (roh): (17007, 18)
Highest Grossing (roh): (104, 5)



## 1. Erster Überblick

Bevor ich Daten bereinige, prüfe ich zuerst:
- die Spaltenstruktur
- Datentypen
- fehlende Werte
- Dubletten
- erste Beispielzeilen


In [2]:

display(Markdown("### Strategy Games – erste Zeilen"))
display(strategy_raw.head(3))

display(Markdown("### Highest Grossing – erste Zeilen"))
display(grossing_raw.head(3))


### Strategy Games – erste Zeilen

,URL,ID,Name,Subtitle,Icon URL,Average User Rating,User Rating Count,Price,In-app Purchases,Description,Developer,Age Rating,Languages,Size,Primary Genre,Genres,Original Release Date,Current Version Release Date
0,https://apps.apple.com/us/app/sudoku/id284921427,284921427,Sudoku,NaN,https://is2-ssl.mzstatic.com/image/thumb/Purpl...,4.0,3553.0,2.99,NaN,"Join over 21,000,000 of our fans and download ...",Mighty Mighty Good Games,4+,"DA, NL, EN, FI, FR, DE, IT, JA, KO, NB, PL, PT...",15853568.0,Games,"Games, Strategy, Puzzle",11/07/2008,30/05/2017
1,https://apps.apple.com/us/app/reversi/id284926400,284926400,Reversi,NaN,https://is4-ssl.mzstatic.com/image/thumb/Purpl...,3.5,284.0,1.99,NaN,"The classic game of Reversi, also known as Oth...",Kiss The Machine,4+,EN,12328960.0,Games,"Games, Strategy, Board",11/07/2008,17/05/2018
2,https://apps.apple.com/us/app/morocco/id284946595,284946595,Morocco,NaN,https://is5-ssl.mzstatic.com/image/thumb/Purpl...,3.0,8376.0,0.00,NaN,Play the classic strategy game Othello (also k...,Bayou Games,4+,EN,674816.0,Games,"Games, Board, Strategy",11/07/2008,5/09/2017


### Highest Grossing – erste Zeilen

,Game,Revenue,Initial release,Publisher(s),Genre(s)
0,Honor of Kings / Arena of Valor,14667500000,2015-11-26,Tencent Games,MOBA
1,Monster Strike,10000000000,2013-08-08,Mixi,Puzzle / RPG / Strategy
2,PUBG Mobile,9000000000,2018-03-19,Tencent Games / Krafton / VNG Games,Battle royale


In [3]:

print("=== Strategy Games: Spalten ===")
print(strategy_raw.columns.tolist())
print("\nExakte Dubletten:", strategy_raw.duplicated().sum())
print("\nFehlende Werte:")
display(strategy_raw.isna().sum().sort_values(ascending=False).to_frame("missing_values").head(12))

print("\n=== Highest Grossing: Spalten ===")
print(grossing_raw.columns.tolist())
print("\nExakte Dubletten:", grossing_raw.duplicated().sum())
print("\nFehlende Werte:")
display(grossing_raw.isna().sum().sort_values(ascending=False).to_frame("missing_values"))


=== Strategy Games: Spalten ===
['URL', 'ID', 'Name', 'Subtitle', 'Icon URL', 'Average User Rating', 'User Rating Count', 'Price', 'In-app Purchases', 'Description', 'Developer', 'Age Rating', 'Languages', 'Size', 'Primary Genre', 'Genres', 'Original Release Date', 'Current Version Release Date']

Exakte Dubletten: 160

Fehlende Werte:


,missing_values
Subtitle,11746
Average User Rating,9446
User Rating Count,9446
In-app Purchases,9324
Languages,60
Price,24
Size,1
URL,0
Original Release Date,0
Genres,0



=== Highest Grossing: Spalten ===
['Game', 'Revenue', 'Initial release', 'Publisher(s)', 'Genre(s)']

Exakte Dubletten: 0

Fehlende Werte:


,missing_values
Game,0
Revenue,0
Initial release,0
Publisher(s),0
Genre(s),0



## 2. Hilfsfunktionen

Diese Funktionen machen die Bereinigung robuster und besser lesbar.


In [4]:

def to_snake_case_columns(columns):
    """Wandelt Spaltennamen in ein konsistentes snake_case-Format um."""
    cleaned = []
    for col in columns:
        col = str(col).strip().lower()
        col = col.replace("&", "and")
        col = re.sub(r"[^a-z0-9]+", "_", col)
        col = re.sub(r"_+", "_", col).strip("_")
        cleaned.append(col)
    return cleaned


def clean_text(value):
    """Trimmt Text und reduziert Mehrfach-Leerzeichen. Fehlende Werte bleiben erhalten."""
    if pd.isna(value):
        return value
    value = str(value).replace("\u00a0", " ")
    value = re.sub(r"\s+", " ", value).strip()
    return value


def split_clean_list(value, sep=","):
    """Zerlegt Listenfelder in bereinigte Python-Listen."""
    if pd.isna(value) or str(value).strip() == "":
        return []
    parts = [clean_text(x) for x in str(value).split(sep)]
    return [x for x in parts if x]


def parse_price_list(value):
    """Wandelt eine kommaseparierte Preisliste in floats um."""
    if pd.isna(value) or str(value).strip() == "":
        return []
    prices = []
    for item in str(value).split(","):
        item = item.strip()
        try:
            prices.append(float(item))
        except ValueError:
            continue
    return prices



## 3. Bereinigung des Datensatzes `17k_strategy_games.csv`

### Was hier gemacht wird
- unnötige Spalten entfernen
- Spaltennamen vereinheitlichen
- Dubletten entfernen
- Textfelder trimmen
- Datumsfelder parsen
- numerische Felder absichern
- abgeleitete Analyse-Features erstellen
- Plausibilitätschecks ergänzen


In [5]:
# Arbeitskopie erstellen
strategy = strategy_raw.copy()

In [6]:
# Spaltennamen vereinheitlichen
strategy.columns = (
    strategy.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [7]:
# Fehlende Werte behandeln
strategy["average_user_rating"] = strategy["average_user_rating"].fillna(0)
strategy["user_rating_count"] = strategy["user_rating_count"].fillna(0)

In [8]:
# Spalten entfernen, die für die Bereinigung/Analyse nicht benötigt werden
drop_cols = ["URL", "ID", "Subtitle", "Icon URL", "Description"]
strategy = strategy.drop(columns=[c for c in drop_cols if c in strategy.columns])

# Spaltennamen vereinheitlichen
strategy.columns = to_snake_case_columns(strategy.columns)

# Exakte Dubletten entfernen
strategy = strategy.drop_duplicates().reset_index(drop=True)

# Textspalten bereinigen
text_cols = strategy.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    strategy[col] = strategy[col].map(clean_text)

# Leere Strings als fehlende Werte markieren
for col in ["name", "in_app_purchases", "developer", "age_rating", "languages", "primary_genre", "genres"]:
    if col in strategy.columns:
        strategy[col] = strategy[col].replace("", pd.NA)

# Datumsfelder parsen
for col in ["original_release_date", "current_version_release_date"]:
    if col in strategy.columns:
        strategy[col] = pd.to_datetime(strategy[col], format="%d/%m/%Y", errors="coerce")

# Numerische Felder absichern
for col in ["average_user_rating", "user_rating_count", "price", "size"]:
    if col in strategy.columns:
        strategy[col] = pd.to_numeric(strategy[col], errors="coerce")

# Abgeleitete Features
strategy["size_mb"] = strategy["size"] / (1024 * 1024)

strategy["in_app_purchase_list"] = strategy["in_app_purchases"].apply(parse_price_list)
strategy["has_in_app_purchases"] = strategy["in_app_purchase_list"].apply(lambda x: len(x) > 0)
strategy["in_app_purchase_count"] = strategy["in_app_purchase_list"].apply(len)
strategy["min_in_app_purchase"] = strategy["in_app_purchase_list"].apply(lambda x: min(x) if len(x) > 0 else pd.NA)
strategy["max_in_app_purchase"] = strategy["in_app_purchase_list"].apply(lambda x: max(x) if len(x) > 0 else pd.NA)
strategy["avg_in_app_purchase"] = strategy["in_app_purchase_list"].apply(lambda x: sum(x) / len(x) if len(x) > 0 else pd.NA)

strategy["language_count"] = strategy["languages"].apply(lambda x: len(split_clean_list(x, sep=",")))
strategy["genre_count"] = strategy["genres"].apply(lambda x: len(split_clean_list(x, sep=",")))

# Plausibilitätsflags
strategy["price_is_valid"] = strategy["price"].isna() | (strategy["price"] >= 0)
strategy["size_is_valid"] = strategy["size"].isna() | (strategy["size"] > 0)
strategy["rating_is_valid"] = strategy["average_user_rating"].isna() | strategy["average_user_rating"].between(0, 5)

print("Shape nach Bereinigung:", strategy.shape)
display(strategy.head(5))


Shape nach Bereinigung: (16847, 30)


,url,id,name,subtitle,icon_url,average_user_rating,user_rating_count,price,in_app_purchases,description,...,has_in_app_purchases,in_app_purchase_count,min_in_app_purchase,max_in_app_purchase,avg_in_app_purchase,language_count,genre_count,price_is_valid,size_is_valid,rating_is_valid
0,https://apps.apple.com/us/app/sudoku/id284921427,284921427,Sudoku,NaN,https://is2-ssl.mzstatic.com/image/thumb/Purpl...,4.0,3553.0,2.99,NaN,"Join over 21,000,000 of our fans and download ...",...,False,0,<NA>,<NA>,<NA>,17,3,True,True,True
1,https://apps.apple.com/us/app/reversi/id284926400,284926400,Reversi,NaN,https://is4-ssl.mzstatic.com/image/thumb/Purpl...,3.5,284.0,1.99,NaN,"The classic game of Reversi, also known as Oth...",...,False,0,<NA>,<NA>,<NA>,1,3,True,True,True
2,https://apps.apple.com/us/app/morocco/id284946595,284946595,Morocco,NaN,https://is5-ssl.mzstatic.com/image/thumb/Purpl...,3.0,8376.0,0.00,NaN,Play the classic strategy game Othello (also k...,...,False,0,<NA>,<NA>,<NA>,1,3,True,True,True
3,https://apps.apple.com/us/app/sudoku-free/id28...,285755462,Sudoku (Free),NaN,https://is3-ssl.mzstatic.com/image/thumb/Purpl...,3.5,190394.0,0.00,NaN,"Top 100 free app for over a year.\nRated ""Best...",...,False,0,<NA>,<NA>,<NA>,17,3,True,True,True
4,https://apps.apple.com/us/app/senet-deluxe/id2...,285831220,Senet Deluxe,NaN,https://is1-ssl.mzstatic.com/image/thumb/Purpl...,3.5,28.0,2.99,NaN,"""Senet Deluxe - The Ancient Game of Life and A...",...,False,0,<NA>,<NA>,<NA>,15,4,True,True,True


In [9]:
# fehlende In-App-Käufe als 0 markieren
strategy["in_app_purchases"] = strategy["in_app_purchases"].replace("", "0").fillna("0")
strategy["average_user_rating"] = strategy["average_user_rating"].fillna(0)
strategy["user_rating_count"] = strategy["user_rating_count"].fillna(0)

In [10]:
display(Markdown("### Qualitätscheck – Strategy Games"))
display(strategy.isna().sum().sort_values(ascending=False).to_frame("missing_values").head(15))

# Dublettenprüfung nur auf hashbaren Spalten
safe_duplicate_cols = []
for col in strategy.columns:
    contains_list = strategy[col].apply(lambda x: isinstance(x, list)).any()
    if not contains_list:
        safe_duplicate_cols.append(col)

print("Verbliebene Dubletten:", strategy[safe_duplicate_cols].duplicated().sum())
print("Ungültige Preise:", (~strategy["price_is_valid"]).sum())
print("Ungültige Größen:", (~strategy["size_is_valid"]).sum())
print("Ungültige Ratings:", (~strategy["rating_is_valid"]).sum())

### Qualitätscheck – Strategy Games

,missing_values
subtitle,11635
avg_in_app_purchase,9232
min_in_app_purchase,9232
max_in_app_purchase,9232
languages,60
price,24
size,1
size_mb,1
current_version_release_date,0
language_count,0


Verbliebene Dubletten: 0
Ungültige Preise: 0
Ungültige Größen: 0
Ungültige Ratings: 0



## 4. Bereinigung des Datensatzes `Highest_Grossing_Mobile_Game.csv`

Dieser Datensatz ist kleiner und sauberer.  
Hier konzentriere ich mich auf:
- Spaltennamen
- Textbereinigung
- Datumsformat
- Umsatz-Felder
- Publisher- und Genre-Auswertung


In [11]:

grossing = grossing_raw.copy()

# Spaltennamen vereinheitlichen
grossing.columns = to_snake_case_columns(grossing.columns)

# Textfelder bereinigen
text_cols = grossing.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    grossing[col] = grossing[col].map(clean_text)

# Datumsfeld parsen
grossing["initial_release"] = pd.to_datetime(grossing["initial_release"], errors="coerce")

# Umsatz numerisch absichern
grossing["revenue"] = pd.to_numeric(grossing["revenue"], errors="coerce")

# Listenbasierte Features
grossing["publisher_list"] = grossing["publisher_s"].apply(lambda x: split_clean_list(x, sep="/"))
grossing["genre_list"] = grossing["genre_s"].apply(lambda x: split_clean_list(x, sep="/"))

grossing["publisher_count"] = grossing["publisher_list"].apply(len)
grossing["genre_count"] = grossing["genre_list"].apply(len)
grossing["revenue_usd_billions"] = grossing["revenue"] / 1_000_000_000
grossing["revenue_is_valid"] = grossing["revenue"].notna() & (grossing["revenue"] >= 0)

print("Shape nach Bereinigung:", grossing.shape)
display(grossing.head(5))


Shape nach Bereinigung: (104, 11)


,game,revenue,initial_release,publisher_s,genre_s,publisher_list,genre_list,publisher_count,genre_count,revenue_usd_billions,revenue_is_valid
0,Honor of Kings / Arena of Valor,14667500000,2015-11-26,Tencent Games,MOBA,[Tencent Games],[MOBA],1,1,14.66750,True
1,Monster Strike,10000000000,2013-08-08,Mixi,Puzzle / RPG / Strategy,[Mixi],"[Puzzle, RPG, Strategy]",1,3,10.00000,True
2,PUBG Mobile,9000000000,2018-03-19,Tencent Games / Krafton / VNG Games,Battle royale,"[Tencent Games, Krafton, VNG Games]",[Battle royale],3,1,9.00000,True
3,Puzzle & Dragons,8578340000,2012-02-20,GungHo Online Entertainment,RPG / Puzzle,[GungHo Online Entertainment],"[RPG, Puzzle]",1,2,8.57834,True
4,Clash of Clans,8000000000,2012-08-02,Supercell (Tencent),Strategy,[Supercell (Tencent)],[Strategy],1,1,8.00000,True


In [12]:
display(Markdown("### Qualitätscheck – Highest Grossing"))
display(grossing.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

# Dublettenprüfung nur auf hashbaren Spalten
safe_duplicate_cols = []

for col in grossing.columns:
    contains_list = grossing[col].apply(lambda x: isinstance(x, list)).any()
    if not contains_list:
        safe_duplicate_cols.append(col)

print("Verbliebene Dubletten:", grossing[safe_duplicate_cols].duplicated().sum())
print("Ungültige Umsatzwerte:", (~grossing["revenue_is_valid"]).sum())


### Qualitätscheck – Highest Grossing

,missing_values
game,0
revenue,0
initial_release,0
publisher_s,0
genre_s,0
publisher_list,0
genre_list,0
publisher_count,0
genre_count,0
revenue_usd_billions,0


Verbliebene Dubletten: 0
Ungültige Umsatzwerte: 0



## 6. Export der bereinigten Dateien

Die Rohdaten bleiben unangetastet.  
Die bereinigten Datensätze werden als neue Dateien gespeichert.


In [13]:

strategy_out = "Datensatz/17k_strategy_games_cleaned.csv"
grossing_out = "Datensatz/Highest_Grossing_Mobile_Game_cleaned.csv"

strategy.to_csv(strategy_out, index=False)
grossing.to_csv(grossing_out, index=False)

print("Gespeichert:")
print(strategy_out)
print(grossing_out)


Gespeichert:
Datensatz/17k_strategy_games_cleaned.csv
Datensatz/Highest_Grossing_Mobile_Game_cleaned.csv
